# Winner XGBoost: 0.95 → UID magic 0.96
[Chris Deotte 공개 코드](https://www.kaggle.com/cdeotte/xgb-fraud-with-magic-0-9600)의 원본 CSV → V 선택 → 인코딩 → UID 집계 → XGB 흐름을 옮긴다.
`baseline/baseline.ipynb`의 choco Model2와 별개 실험이다. 원문 제목의 0.9600은 공개 LB 설명이며 여기서 측정한 점수가 아니다.

XGB96을 기본으로 실행한다. `BUILD95=True`이면 UID 집계 전 모델도 같은 방식으로 비교한다.
최종 CAT/LGB/XGB 전체 학습을 재현한다고 주장하지 않는다. 공개 모델 예측 앙상블은 `winner_blend`에 분리했다.
학습 입력은 원본 CSV 4개뿐이다. UID 후처리는 원문이 사용한 별도 공개 UID CSV 2개를 추가로 읽는다.
우승자의 공개 CatBoost/LightGBM baseline 학습은 winner_catboost/winner_lgbm 노트북에 별도로 추가했다.

## 이 노트북을 읽는 방법

코드를 실행하기 전에 바로 위의 설명을 읽어보세요. **어떤 질문을 푸는지 → 작은 예시로 계산 → 실제 코드의 변수와 연결 → 출력 해석** 순서로 설명합니다.
코드 아래의 관찰은 이미 저장된 실행 결과를 읽는 안내입니다. '해볼 실험'은 아직 실행하지 않은 제안이며, 실제 결과와 구분했습니다.

처음 읽을 때 함수 이름을 모두 외울 필요는 없습니다. 새 피처를 만날 때마다 **'이 숫자는 무엇을 요약하며, 예측할 때도 알 수 있는가?'**를 물어보세요.
EDA에서 찾은 차이가 모델 성능 개선을 뜻하지는 않습니다. 실험 노트북에서는 **검증 데이터를 정한 뒤 구성 요소 하나씩 비교**해야 개선의 근거를 얻습니다.

처음에는 `01_eda_report` → GitHub `baseline`을 읽고, 이후 `02_winner_eda` → `winner_xgb` → `winner_lgbm` → `winner_catboost` → `winner_blend`로 이어가세요.
우승자 공개 baseline과 최종 우승 제출 전체는 구분합니다.

설명을 보강하면서 학습 코드·기존 표·그래프·실행 범위를 유지했습니다. 이 노트북의 **저장된 출력**과 설정의 **다음 실행 기본값**이 다를 수 있으므로 첫 실행 범위와 metrics를 먼저 확인하세요.


### 처음 만나는 용어는 여기서 잠깐 확인하세요

| 용어 | 여기서 뜻하는 것 |
|---|---|
| 피처(feature) | 모델에 입력할 거래의 정보. 원본 열과 새로 계산한 열 모두 포함 |
| NaN / 결측 | 값이 관측되지 않음. 실제 숫자 0과 다른 상태 |
| fold / validation | 교차검증의 한 분할 / 그 분할에서 평가용으로 제외한 데이터 |
| OOF | 각 train 행을 그 행 없이 학습한 모델로 예측해서 모은 값 |
| smoke | 전체 학습 전에 축소 데이터·rounds로 실행 흐름을 확인하는 실험 |
| leaf | tree에서 조건을 따라 내려간 끝의 구역. 그 구역에 들어온 행에 같은 보정을 줌 |

**제거 실험(ablation)**은 피처나 기법 하나를 뺀 모델을 같은 검증에서 비교하는 방법입니다. '있을 때 좋았으니 도움이 된다'에서 한 걸음 더 나아가 실제 기여를 확인하려는 실험입니다.

## 1. 설정과 원본 CSV

### 왜 원본의 앞 30,000행만 자르지 않을까요?

거래가 시간순이면 앞부분은 특정 월에 몰립니다. 이 노트북은 월별 검증을 하므로 일부 월만 읽으면 검증 설계까지 바뀝니다.
`groupby(month).sample(frac=...)`는 각 월에서 같은 비율로 뽑고 ID 순서로 돌립니다. 이는 정답 비율을 맞춘 stratified sampling은 아닙니다. 월의 비중을 대략 유지하는 표본입니다.
월별 반올림 때문에 요청 30,000과 실제 행 수가 조금 다를 수 있습니다. 로그의 train/test와 월별 건수가 실제 범위입니다.

`usecols`는 원문의 거래 기본 열과 V120만 읽습니다. 이미 저장된 전처리 입력을 읽는 것이 아니라 **원본 CSV에서 읽을 열을 제한하는 것**입니다.
identity는 ID로 붙이고 ID는 정수로 유지합니다. 원본 V 선택은 새 EDA의 자동 결과가 아닌 작성자가 정한 목록입니다.
`y_train=pop('isFraud')`는 정답을 입력에서 빼고 따로 보관합니다. 특징 집계 함수가 정답을 입력으로 받지 않는지 확인해보세요.

In [1]:
import gc, json, logging, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'source.json').exists())
sys.path.insert(0, str(ROOT))
from data.loader import get_data_dir
from data.winner import V_KEEP, get_released_dir, postprocess

NROWS = int(os.environ['IEEE_WINNER_NROWS']) if os.getenv('IEEE_WINNER_NROWS') else None
FOLDS = int(os.getenv('IEEE_WINNER_FOLDS', '6'))
ROUNDS = int(os.getenv('IEEE_WINNER_ROUNDS', '5000'))
LOCAL_ROUNDS = int(os.getenv('IEEE_WINNER_LOCAL_ROUNDS', '2000'))
DEVICE = os.getenv('IEEE_WINNER_DEVICE', 'cuda')
BUILD95, RUN_LOCAL, RUN_PP = False, True, True
TAG = os.getenv('IEEE_WINNER_TAG', 'smoke' if NROWS else 'full')
OUT = ROOT / 'experiments/winner_xgb/outputs' / TAG
OUT.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger('winner_xgb')
logger.handlers.clear()
logger.setLevel(logging.INFO)
logger.addHandler(logging.FileHandler(OUT / 'run.log', mode='w', encoding='utf-8'))
logger.addHandler(logging.StreamHandler(sys.stdout))
started = time.perf_counter()

def month(frame):
    dates = pd.Timestamp('2017-11-30') + pd.to_timedelta(frame.TransactionDT, unit='s')
    return (dates.dt.year - 2017) * 12 + dates.dt.month

categorical = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']
categorical += [f'M{i}' for i in range(1, 10)]
categorical += [f'id_{i:02}' for i in [12, 15, 16, 23, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38]]
categorical += ['DeviceType', 'DeviceInfo']
raw = get_data_dir()
base = pd.read_csv(raw / 'train_transaction.csv', nrows=0).columns
usecols = [c for c in base if c not in ['isFraud'] and (not c.startswith('V') or int(c[1:]) in V_KEEP)]

def read_raw(split):
    cols = usecols + (['isFraud'] if split == 'train' else [])
    dtype = {c: 'category' if c in categorical else 'float32' for c in cols}
    dtype['TransactionID'] = 'int64'
    frame = pd.read_csv(raw / f'{split}_transaction.csv', usecols=cols, dtype=dtype).set_index('TransactionID')
    if NROWS:
        frame = frame.groupby(month(frame), group_keys=False).sample(frac=min(1, NROWS / len(frame)), random_state=42).sort_index()
    id_cols = pd.read_csv(raw / f'{split}_identity.csv', nrows=0).columns
    id_dtype = {c: 'category' if c.replace('-', '_') in categorical else 'float32' for c in id_cols}
    id_dtype['TransactionID'] = 'int64'
    identity = pd.read_csv(raw / f'{split}_identity.csv', dtype=id_dtype).set_index('TransactionID')
    identity.columns = identity.columns.str.replace('-', '_', regex=False)
    assert frame.index.is_unique and identity.index.is_unique
    return frame.join(identity, how='left', validate='one_to_one')

X_train, X_test = read_raw('train'), read_raw('test')
y_train = X_train.pop('isFraud').astype('int8')
groups = month(X_train)
assert X_train.TransactionDT.is_monotonic_increasing
logger.info('train=%s test=%s monthly rows=%s device=%s', X_train.shape, X_test.shape, groups.value_counts().sort_index().to_dict(), DEVICE)

train=(30000, 213) test=(30002, 213) monthly rows={12: 6976, 13: 4703, 14: 4370, 15: 5163, 16: 4250, 17: 4538} device=cuda


## 2. 원본 코드의 인코딩과 0.95 피처

### encode 함수 5개를 각각 하나의 질문으로 읽습니다

| 함수 | 거래에 붙이는 정보 | 간단 예시 |
|---|---|---|
| encode_LE | 범주의 숫자 이름표 | A/B/A→0/1/0 |
| encode_FE | 범주의 전체 등장 비율 | A가 10건 중 3건→.3 |
| encode_CB | 두 열의 공동 경우 | card A,addr X→A_X |
| encode_AG | 그룹의 mean/std | 금액 10/10/40→평균20,std17.32 |
| encode_AG2 | 그룹 안의 서로 다른 값 수 | 이메일 A/A/B→2 |

이 XGB의 FE는 **비율**이고 GitHub Model2의 빈도는 **건수**입니다. 같은 이름의 기법이라도 실제 연산을 확인해야 합니다.
각 함수는 train+test 공동으로 계산합니다. 검증 입력도 통계에 포함되지만 정답은 쓰지 않습니다. 그래서 검증도 전체 배치를 보는 조건이며 train-only 집계 실험과 구분해야 합니다.

D4 등은 `D-day`로 바꿉니다. day=100,D4=20과 day=105,D4=25는 둘 다 -80입니다. 시간 경과 때문에 같이 증가하는 값을 '시작 시점의 음수' 형태로 고정하려는 아이디어입니다.
이후 금액/시간을 제외한 숫자는 공동 최소값을 빼므로 순서는 유지되고 결측은 -1이 됩니다. **tree가 반드시 양수 입력을 요구해서 하는 처리는 아닙니다.** 원문 표현과 sentinel 규칙을 유지하는 단계입니다.

`usena=True`는 -1을 집계에서 빼고, singleton std/없는 mapping도 -1로 둡니다. mean [10,-1,30]을 그대로 구하면 13이지만 -1을 결측으로 빼면 20입니다.
`AG2.nunique()`의 -1은 현재 구현에서 하나의 서로 다른 값으로 포함됩니다. 평균 집계와 고유값 집계의 결측 규칙이 같지 않습니다.
`cents`는 금액의 소수 부분입니다. 카드×주소×이메일 조합과 그 금액/D9/D11 평균으로 UID 이전의 0.95 피처를 만듭니다.

Label encoding된 범주는 여기서 XGB에 **숫자 입력**으로 들어갑니다. 이 구현은 native categorical split을 켜지 않았으므로 숫자 코드의 임의 순서에 민감할 수 있습니다. 공동 인코딩은 train/test 코드 불일치를 막는 것부터 해결합니다.

In [2]:
def encode_LE(col):
    values, _ = pd.concat([X_train[col], X_test[col]]).factorize(sort=True)
    dtype = 'int32' if values.max() > 32000 else 'int16'
    X_train[col] = values[:len(X_train)].astype(dtype)
    X_test[col] = values[len(X_train):].astype(dtype)

def encode_FE(cols):
    for col in cols:
        counts = pd.concat([X_train[col], X_test[col]]).value_counts(normalize=True).to_dict()
        counts[-1] = -1
        for frame in [X_train, X_test]:
            frame[col + '_FE'] = frame[col].map(counts).astype('float32')

def encode_CB(col1, col2):
    name = col1 + '_' + col2
    for frame in [X_train, X_test]:
        frame[name] = frame[col1].astype(str) + '_' + frame[col2].astype(str)
    encode_LE(name)

def encode_AG(cols, uids, stats=('mean',), usena=False):
    for uid in uids:
        for col in cols:
            combined = pd.concat([X_train[[uid, col]], X_test[[uid, col]]])
            if usena:
                combined[col] = combined[col].astype('float32').mask(combined[col].eq(-1))
            for stat in stats:
                mapping = combined.groupby(uid)[col].agg(stat)
                for frame in [X_train, X_test]:
                    frame[f'{col}_{uid}_{stat}'] = frame[uid].map(mapping).astype('float32').fillna(-1)

def encode_AG2(cols, uids):
    for uid in uids:
        for col in cols:
            combined = pd.concat([X_train[[uid, col]], X_test[[uid, col]]])
            mapping = combined.groupby(uid)[col].nunique()
            for frame in [X_train, X_test]:
                frame[f'{uid}_{col}_ct'] = frame[uid].map(mapping).astype('float32')

for frame in [X_train, X_test]:
    for i in range(1, 16):
        if i not in [1, 2, 3, 5, 9]:
            frame[f'D{i}'] -= frame.TransactionDT / np.float32(86400)
for col in X_train:
    if isinstance(X_train[col].dtype, pd.CategoricalDtype):
        encode_LE(col)
    elif col not in ['TransactionAmt', 'TransactionDT']:
        minimum = np.min([X_train[col].min(), X_test[col].min()])
        for frame in [X_train, X_test]:
            frame[col] = (frame[col] - np.float32(minimum)).fillna(-1)
X_train, X_test = X_train.copy(), X_test.copy()
for frame in [X_train, X_test]:
    frame['cents'] = (frame.TransactionAmt - np.floor(frame.TransactionAmt)).astype('float32')
encode_FE(['addr1', 'card1', 'card2', 'card3', 'P_emaildomain'])
encode_CB('card1', 'addr1')
encode_CB('card1_addr1', 'P_emaildomain')
encode_FE(['card1_addr1', 'card1_addr1_P_emaildomain'])
encode_AG(['TransactionAmt', 'D9', 'D11'], ['card1', 'card1_addr1', 'card1_addr1_P_emaildomain'], ('mean', 'std'), usena=True)
dropped = ['TransactionDT', 'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14', 'C3', 'M5', 'id_08', 'id_33',
           'card4', 'id_07', 'id_14', 'id_21', 'id_30', 'id_32', 'id_34'] + [f'id_{i}' for i in range(22, 28)]
cols95 = [c for c in X_train if c not in dropped]
for frame in [X_train, X_test]:
    frame['DT_M'] = month(frame)

## 3. UID magic와 0.96 피처

### UID magic은 '같은 사람을 맞힌다'보다 '그룹의 특징을 준다'에 가깝습니다

날짜가 다른 거래라도 card1_addr1과 `floor(day-D1)`이 같으면 같은 UID로 묶습니다. 예를 들어 100일/20일, 105일/25일은 시작일 80이 같습니다.
코드의 D1은 앞 셀의 최소값 이동/결측 치환을 거친 값입니다. raw D1 공식을 설명하는 예시와 실제 문자열을 완전히 같은 것으로 읽지 마세요. 특히 결측 -1은 실제 카드 시작일이 아닙니다.

UID 하나가 여러 실제 고객을 섞을 수 있어 여러 관점을 붙입니다.
**금액/D의 mean/std**는 금액 규모와 시작 시점 단서의 일관성, **C mean**은 활동량, **M mean**은 관측된 일치 성향을 요약합니다.
**이메일/월/금액소수/V의 nunique**는 그룹의 다양성을 측정합니다. 같은 UID의 이메일이 1개인지 10개인지에 따라 그룹의 균질함이 달라 보일 수 있습니다.

M 값 1/0/-1에서 -1을 제외하고 평균내면 관측된 T 비율입니다. C14의 std는 평균과 별개로 변화량을 표현합니다.
`outsider15=abs(D1-D15)>3`은 두 시간차 단서가 크게 다른지 나타내는 휴리스틱입니다. 앞서 D15는 day 차감/최소값 이동을 거쳤으므로 **이 코드가 raw D1과 raw D15만 비교한다고 설명하면 틀립니다.** 원문의 변환된 피처 비교입니다.

모델에는 `uid` 문자열 자체를 넣지 않고 집계만 넣습니다. 처음 보는 UID도 그 거래들의 구조를 표현하도록 하려는 선택입니다.
EDA/PP의 정밀 UID 파일은 이 집계를 만드는 입력이 아닙니다. 이 셀에서 만든 단순 UID와 정밀 UID의 역할을 섞지 마세요.

In [3]:
X_train, X_test = X_train.copy(), X_test.copy()
for frame in [X_train, X_test]:
    frame['day'] = frame.TransactionDT / 86400
    frame['uid'] = frame.card1_addr1.astype(str) + '_' + np.floor(frame.day - frame.D1).astype(str)
encode_FE(['uid'])
encode_AG(['TransactionAmt', 'D4', 'D9', 'D10', 'D15'], ['uid'], ('mean', 'std'), usena=True)
encode_AG([f'C{i}' for i in range(1, 15) if i != 3], ['uid'], usena=True)
encode_AG([f'M{i}' for i in range(1, 10)], ['uid'], usena=True)
encode_AG2(['P_emaildomain', 'dist1', 'DT_M', 'id_02', 'cents'], ['uid'])
encode_AG(['C14'], ['uid'], ('std',), usena=True)
encode_AG2(['C13', 'V314', 'V127', 'V136', 'V309', 'V307', 'V320'], ['uid'])
for frame in [X_train, X_test]:
    frame['outsider15'] = (np.abs(frame.D1 - frame.D15) > 3).astype('int8')
cols96 = [c for c in X_train if c not in dropped + ['DT_M', 'day', 'uid']]
logger.info('V=%d features95=%d features96=%d', len(V_KEEP), len(cols95), len(cols96))
assert 'uid' not in cols96 and 'isFraud' not in cols96
display(pd.Series({'XGB95': len(cols95), 'XGB96': len(cols96)}, name='features'))

V=120 features95=216 features96=263


XGB95    216
XGB96    263
Name: features, dtype: int64

### 피처 수가 보여주는 것
저장된 피처 수는 UID 전 216개, 후 263개입니다. 개수가 늘었다는 것 자체가 성능 향상을 뜻하지 않습니다.
**해볼 실험:** BUILD95를 켜 같은 데이터/검증/rounds에서 두 모델을 비교하세요. 원문의 보고 개선과 로컬의 개선은 별개이며 현재 실행은 그 비교를 수행하지 않았습니다.

## 4. 시간순 holdout와 월별 CV

### 먼저, 여러 tree를 왜 순서대로 더할까요?

한 tree는 '금액이 크고, 특정 기기이고, 어떤 카드 그룹인가'처럼 조건을 나눠 거래를 구분합니다. 여러 조건의 **상호작용**을 표현할 수 있습니다.
Gradient boosting은 새 tree를 독립적으로 평균내는 방식과 다릅니다. 앞의 예측이 놓친 방향을 다음 tree가 보정합니다.

이진 분류에서 예측 확률 p와 정답 y의 log loss는 `-[y log(p)+(1-y)log(1-p)]`입니다.
사기인데 p=.1이면 큰 벌점을 받고, 사기인데 p=.9이면 작은 벌점을 받습니다. logit 점수에 대한 음의 기울기는 `y-p`입니다.
사기 y=1,p=.1은 +.9 방향, 정상 y=0,p=.9는 -.9 방향으로 보정하도록 새 tree를 학습합니다. 실제 leaf 값은 여러 행의 기울기/가중치 등을 모아 구합니다.

개념적으로 `새 점수 = 이전 점수 + learning_rate × 새 tree 보정`이며 확률은 점수를 sigmoid로 바꿔 얻습니다.
작은 learning rate는 조금씩 고치는 대신 더 많은 tree가 필요합니다. **rounds를 크게 적었다고 반드시 그 수만큼 학습하는 것도, 최적 튜닝을 마친 것도 아닙니다.**

여기서 loss는 학습 방향을 정하고 AUC는 검증에서 순위를 평가합니다. **AUC를 지표로 적었어도 tree가 직접 AUC를 미분해 학습하는 것은 아닙니다.**
트리의 깊이/leaf 수는 복잡도, 행/열 표본 추출은 다양성과 과적합을, early stopping은 학습을 끝낼 시점을 조절합니다.
이 설명의 수식은 원리를 이해하기 위한 것이며 실제 새 tree는 단일 거래의 오차만으로 만들어지지 않습니다.

여기서 gradient(기울기)는 loss를 줄이려면 예측을 어느 방향으로 고칠지 알려줍니다.
확률 p를 `log(p/(1-p))`로 바꾼 점수를 **logit**이라 부릅니다. p=.5이면 0, p=.9이면 약 2.20입니다.
**Sigmoid**는 `1/(1+exp(-점수))`로 점수를 다시 0~1 확률로 바꿉니다. Tree 보정은 이 점수 공간에서 더하고 마지막에 확률로 읽습니다.


### XGBoost와 현재 파라미터를 연결하기

XGBoost는 tree 분기의 후보를 loss 감소와 정규화 비용으로 비교합니다. Logistic loss에서는 기울기 `g=p-y`, 두 번째 미분 `h=p(1-p)`를 활용합니다.
정규화가 있는 leaf의 보정은 개념적으로 `-Σg/(Σh+λ)`처럼 정해집니다. 모든 오차를 무조건 크게 보정하기보다 안정적으로 보정하려는 구조입니다.
`hist`는 연속값을 bin으로 묶어 분기 탐색을 빠르게 합니다. GPU는 계산 장치이며 다른 새로운 모델 이름이 아닙니다.

| 설정 | 역할 | 다음 실험의 질문 |
|---|---|---|
| max_depth=12 | 최대 분기 경로 길이 | 8/10/12로 낮추면 월별 성능이 안정되나? |
| learning_rate=.02 | 보정의 크기 | 더 작은 값에는 충분한 rounds가 필요한가? |
| subsample=.8 | tree 학습 행 일부 사용 | 희귀 사기를 포함한 표본 변화의 영향은? |
| colsample_bytree=.4 | tree별 열 일부 사용 | UID 집계에만 의존하는 것을 줄이나? |
| missing=-1 | -1을 누락으로 선언 | 단순히 가장 작은 값으로 취급하는 것과 다름 |

위 후보는 **아직 실행하지 않은 튜닝 질문**입니다. 원문 설정이 모든 검증에서 최적은 아닙니다. [tree 원리](https://xgboost.readthedocs.io/en/stable/tutorials/model.html)와 [파라미터 정의](https://xgboost.readthedocs.io/en/stable/parameter.html)를 참고하세요.

### fold, OOF, test 평균을 구분해서 읽기

Fold는 교차검증의 한 번의 학습/평가 분할입니다. 월별 6-fold이면 매번 한 월을 검증에 두고 나머지 월로 학습합니다.
검증 월의 행은 자기 정답을 학습하지 않은 모델에서 예측을 받습니다. 그 값을 원래 행 위치에 모은 것이 **OOF(out-of-fold)**입니다.
OOF를 합쳐 AUC를 구하면 학습 행 재예측보다 일반화 평가에 가깝습니다. 다만 여기서는 early stopping도 같은 validation을 보므로 완전히 손대지 않은 최종 평가셋은 아닙니다.

`pred += fold_test_pred / FOLDS`는 test를 각 fold 모델로 예측해 평균냅니다. test에는 정답이 없어 AUC를 계산할 수 없습니다.
GroupKFold의 groups는 월입니다. **같은 월의 행이 fit/validation 양쪽에 들어가지는 않지만 같은 UID의 다른 월 거래는 들어갈 수 있습니다.**
또 검증이 과거 월이면 이후 월도 fit에 들어갑니다. 그러므로 이 검증을 미래 예측이나 고객 전체 미관측 검증으로 부르면 안 됩니다.
미래 예측은 시간 holdout, 새 고객 예측은 UID 그룹 분할 등으로 따로 질문해야 합니다. [GroupKFold 정의](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)를 참고하세요.

### 이 노트북의 두 검증

시간 holdout은 앞 75%로 학습해 뒤 25%를 예측합니다. `is_monotonic_increasing`으로 시간 순서 전제를 확인합니다.
월별 CV는 한 월을 통째로 제외하고 다른 월로 학습합니다. holdout과 숫자가 다르면 어떤 질문/학습량이 다른지 먼저 봅니다.
Early stopping은 holdout 100, CV 200 rounds의 개선 정체를 봅니다. 현재 저장된 100-round smoke는 upper bound에 먼저 닿을 수 있어 장기 튜닝 결과가 아닙니다.
`predict`는 best_iteration까지의 tree만 사용합니다. `BUILD95=True`이면 같은 분할에서 cols95/cols96를 비교할 수 있지만 현재 저장된 출력은 XGB96만 실행한 것입니다.

In [4]:
params = dict(max_depth=12, learning_rate=0.02, subsample=0.8, colsample_bytree=0.4,
              missing=-1, eval_metric='auc', tree_method='hist', device=DEVICE, n_jobs=8, random_state=0)
def predict(model, frame):
    matrix = xgb.DMatrix(frame, missing=-1, nthread=8)
    return model.get_booster().predict(matrix, iteration_range=(0, model.best_iteration + 1))

results, predictions = {}, {}
feature_sets = {'xgb95': cols95, 'xgb96': cols96} if BUILD95 else {'xgb96': cols96}
for name, features in feature_sets.items():
    train_matrix, test_matrix = X_train[features], X_test[features]
    local_auc = None
    if RUN_LOCAL:
        cut = 3 * len(X_train) // 4
        model = xgb.XGBClassifier(**params, n_estimators=LOCAL_ROUNDS, early_stopping_rounds=100)
        model.fit(train_matrix.iloc[:cut], y_train.iloc[:cut], eval_set=[(train_matrix.iloc[cut:], y_train.iloc[cut:])], verbose=False)
        local_auc = float(roc_auc_score(y_train.iloc[cut:], predict(model, train_matrix.iloc[cut:])))
        logger.info('%s time holdout AUC=%.6f best_iteration=%d', name, local_auc, model.best_iteration)
        del model
    oof, test_pred = np.full(len(X_train), np.nan), np.zeros(len(X_test))
    folds, importance = [], np.zeros(len(features))
    splitter = GroupKFold(n_splits=FOLDS)
    for fold, (fit_idx, val_idx) in enumerate(splitter.split(train_matrix, y_train, groups)):
        assert set(groups.iloc[fit_idx]).isdisjoint(groups.iloc[val_idx])
        model = xgb.XGBClassifier(**params, n_estimators=ROUNDS, early_stopping_rounds=200)
        model.fit(train_matrix.iloc[fit_idx], y_train.iloc[fit_idx], eval_set=[(train_matrix.iloc[val_idx], y_train.iloc[val_idx])], verbose=False)
        oof[val_idx] = predict(model, train_matrix.iloc[val_idx])
        test_pred += predict(model, test_matrix) / FOLDS
        importance += model.feature_importances_ / FOLDS
        score = float(roc_auc_score(y_train.iloc[val_idx], oof[val_idx]))
        folds.append({'fold': fold, 'months': sorted(map(int, groups.iloc[val_idx].unique())), 'auc': score, 'best_iteration': int(model.best_iteration)})
        logger.info('%s fold=%d months=%s AUC=%.6f best_iteration=%d', name, fold, folds[-1]['months'], score, model.best_iteration)
        del model
        gc.collect()
    assert np.isfinite(oof).all() and np.isfinite(test_pred).all()
    assert np.all((oof >= 0) & (oof <= 1)) and np.all((test_pred >= 0) & (test_pred <= 1))
    score = float(roc_auc_score(y_train, oof))
    predictions[name] = pd.Series(test_pred, index=X_test.index, name='isFraud')
    results[name] = {'features': len(features), 'local_auc': local_auc, 'oof_auc': score, 'folds': folds}
    np.save(OUT / f'{name}_oof.npy', oof)
    np.save(OUT / f'{name}_pred_test.npy', test_pred)
    pd.DataFrame({'isFraud': y_train, 'oof': oof, 'month': groups}).to_csv(OUT / f'{name}_oof.csv')
    pd.Series(importance, index=features, name='importance').sort_values(ascending=False).to_csv(OUT / f'{name}_importance.csv')
    logger.info('%s pooled OOF AUC=%.6f', name, score)
display(pd.DataFrame(results).T[['features', 'local_auc', 'oof_auc']])

xgb96 time holdout AUC=0.860963 best_iteration=41


xgb96 fold=0 months=[12] AUC=0.818325 best_iteration=89


xgb96 fold=1 months=[15] AUC=0.881375 best_iteration=99


xgb96 fold=2 months=[13] AUC=0.848890 best_iteration=98


xgb96 fold=3 months=[17] AUC=0.889826 best_iteration=88


xgb96 fold=4 months=[14] AUC=0.900469 best_iteration=96


xgb96 fold=5 months=[16] AUC=0.884413 best_iteration=84


xgb96 pooled OOF AUC=0.862225


,features,local_auc,oof_auc
xgb96,263,0.860963,0.862225


### 현재 실행의 관찰
월별 30,000 train/30,002 test, 6-fold/100 rounds의 pooled OOF AUC는 .862225, 75/25 시간 holdout은 .860963입니다.
일부 월의 AUC가 더 낮습니다. 피처의 시간 변화인지, 그 월의 고객/상품 구성이 다른지 조사할 질문입니다. 전체 test/원문 .96 LB 재현 점수로 읽지 않습니다.
**학습 순서:** 검증 고정 → UID FE 전/후 → 깊이/정규화 → learning rate와 rounds. 여러 설정을 동시에 바꾸면 개선 이유를 알기 어렵습니다.

## 5. 제출과 공개 UID 후처리

### 정밀 UID로 예측을 평균내면 왜 도움이 될 수 있을까요?

같은 고객의 모든 거래가 같은 정답을 공유한다는 가정이 맞고 예측에 개별 잡음이 있다면, 그룹 평균이 흔들림을 줄일 수 있습니다.
같은 UID의 test 예측 [.2,.8]만 평균내면 둘 다 .5입니다. train에서 그 UID의 정상 정답 0을 한 건 알고 있다면 `[0,.2,.8]` 평균 1/3을 test 두 건에 부여합니다.
**UID 복원이 정확하고 고객의 정답이 일관되어야 하는 기법**입니다. 정상/사기가 섞인 실제 그룹이면 평균이 잘 맞춘 거래를 망칠 수도 있습니다.

원문 PP는 공개 v4→v1 순서입니다. 첫 단계는 train 값까지 갱신하고 두 번째는 그 갱신값을 다시 평균냅니다. 따라서 순서와 파일 버전이 바뀌면 결과도 달라질 수 있습니다.
매칭되지 않는 ID는 기존 예측을 유지합니다. 공개 파일의 커버리지는 100%가 아니므로 단순 inner join은 거래를 잃습니다.

이 코드에서 PP는 전체 학습의 test 제출에만 적용합니다. OOF를 PP할 때 validation 정답까지 평균에 섞으면 그 정답을 본 예측이 되어 평가가 무너집니다.
OOF PP를 비교하려면 fold별 fit 라벨만 known label로 사용해야 하고 UID 복원 시 정답을 사용했는지도 점검해야 합니다. **여기서는 그 별도 PP 검증을 실행하지 않았습니다.**

In [5]:
pp_coverage = []
for name, pred in predictions.items():
    if NROWS:
        pred.to_csv(OUT / f'{name}_sample_predictions.csv')
        continue
    sample = pd.read_csv(raw / 'sample_submission.csv').set_index('TransactionID')
    assert set(sample.index) == set(pred.index) and sample.index.is_unique
    pred.reindex(sample.index).to_csv(OUT / f'{name}_submission.csv')
    if RUN_PP and name == 'xgb96':
        processed, pp_coverage = postprocess(y_train, pred, get_released_dir())
        processed.reindex(sample.index).to_csv(OUT / f'{name}_submission_pp.csv')
metrics = {'scope': 'monthly sampled smoke' if NROWS else 'full raw CSV', 'requested_nrows': NROWS,
           'train_rows': len(X_train), 'test_rows': len(X_test), 'folds': FOLDS, 'rounds': ROUNDS,
           'local_rounds': LOCAL_ROUNDS, 'build95': BUILD95, 'params': params, 'results': results,
           'pp_coverage': pp_coverage, 'elapsed_seconds': time.perf_counter() - started,
           'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'xgboost': xgb.__version__}}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
logger.info('saved %s elapsed=%.1fs scope=%s', OUT, metrics['elapsed_seconds'], metrics['scope'])
display(metrics)

saved C:\Users\jiho\code\kaggle-study\IEEE-Fraud-Detection\experiments\winner_xgb\outputs\smoke elapsed=17.7s scope=monthly sampled smoke


{'scope': 'monthly sampled smoke',
 'requested_nrows': 30000,
 'train_rows': 30000,
 'test_rows': 30002,
 'folds': 6,
 'rounds': 100,
 'local_rounds': 100,
 'build95': False,
 'params': {'max_depth': 12,
  'learning_rate': 0.02,
  'subsample': 0.8,
  'colsample_bytree': 0.4,
  'missing': -1,
  'eval_metric': 'auc',
  'tree_method': 'hist',
  'device': 'cuda',
  'n_jobs': 8,
  'random_state': 0},
 'results': {'xgb96': {'features': 263,
   'local_auc': 0.8609629825673552,
   'oof_auc': 0.862224886856649,
   'folds': [{'fold': 0,
     'months': [12],
     'auc': 0.8183254778780242,
     'best_iteration': 89},
    {'fold': 1,
     'months': [15],
     'auc': 0.8813749997720854,
     'best_iteration': 99},
    {'fold': 2,
     'months': [13],
     'auc': 0.848889849167644,
     'best_iteration': 98},
    {'fold': 3,
     'months': [17],
     'auc': 0.8898263989913683,
     'best_iteration': 88},
    {'fold': 4, 'months': [14], 'auc': 0.90046918767507, 'best_iteration': 96},
    {'fold': 5,


## 변경과 재현 범위
원문 셀 3/4/7/9/11/13/15/22/29/31/32/36/39/45의 피처·검증·PP를 따른다. 반복 플롯과 중복 학습을 줄이고 BUILD95를 기본 해제했다.
pandas 3의 제거된 `np.str`·inplace 동작과 XGBoost GPU/early-stopping API를 수정했다. 카테고리 32,000개 초과 시 원문의 경고만 출력하던 int16 대신 int32를 사용한다.
TransactionID는 정수로 보존하고 출력 ID를 검증한다. 현재 라이브러리의 기본값/학습 구현이 2019년과 달라 원문 LB 수치의 완전 일치는 보장하지 않는다.
train+test 집계와 전체 거래를 이용한 UID PP는 대회 배치 조건이다. 원본/raw CSV와 별도 공개 UID 버전을 함께 기록한다.